In [2]:
import pandas as pd
import numpy as np

# Load the processed dataset
df = pd.read_csv(
    "../data/processed/uac_features.csv",
    parse_dates=["Date"]
)

# Sort by date
df = df.sort_values("Date").reset_index(drop=True)

# Make sure total_system_load exists
if "total_system_load" not in df.columns:
    df["total_system_load"] = (
        df["Children in CBP custody"] +
        df["Children in HHS Care"]
    )

# Make sure net_intake exists
if "net_intake" not in df.columns:
    df["net_intake"] = (
        df["Children transferred out of CBP custody"]
        - df["Children discharged from HHS Care"]
    )

# Calculate high-load threshold
load_threshold = df["total_system_load"].quantile(0.75)

# Identify high-load periods
df["high_load"] = (
    df["total_system_load"] >= load_threshold
)

# Identify positive intake pressure
df["positive_intake_pressure"] = (
    df["net_intake"] > 0
)

# Combined pressure indicator
df["load_pressure"] = (
    df["high_load"] &
    df["positive_intake_pressure"]
)

# Show results
print("Load Pressure Threshold:", round(load_threshold, 2))
print("High-load observations:", df["high_load"].sum())
print("Positive intake observations:", df["positive_intake_pressure"].sum())
print("Pressure observations:", df["load_pressure"].sum())

Load Pressure Threshold: 8249.5
High-load observations: 180
Positive intake observations: 238
Pressure observations: 24


In [3]:
pressure_records = df[df["load_pressure"]].copy()

pressure_records[
    [
        "Date",
        "total_system_load",
        "net_intake",
        "high_load",
        "positive_intake_pressure",
        "load_pressure"
    ]
].head(20)

,Date,total_system_load,net_intake,high_load,positive_intake_pressure,load_pressure
21,2023-02-21,8281.0,4.0,True,True,True
130,2023-08-01,8349.0,5.0,True,True,True
250,2024-01-30,8380.0,21.0,True,True,True
254,2024-02-05,8338.0,103.0,True,True,True
255,2024-02-06,8500.0,145.0,True,True,True
256,2024-02-07,8613.0,108.0,True,True,True
257,2024-02-08,8687.0,43.0,True,True,True
259,2024-02-12,8522.0,206.0,True,True,True
260,2024-02-13,8717.0,189.0,True,True,True
261,2024-02-14,8834.0,42.0,True,True,True


In [4]:
import plotly.express as px

fig = px.scatter(
    df,
    x="Date",
    y="total_system_load",
    color="load_pressure",
    title="System Load Pressure Analysis",
    labels={
        "total_system_load": "Total System Load",
        "load_pressure": "Pressure Period"
    }
)

fig.show()

In [5]:
load_threshold = df["total_system_load"].quantile(0.75)

df["high_load"] = (
    df["total_system_load"] >= load_threshold
)

In [6]:
df["positive_intake_pressure"] = (
    df["net_intake"] > 0
)

In [7]:
df["load_pressure"] = (
    df["high_load"] &
    df["positive_intake_pressure"]
)

In [8]:
pressure_records = df[
    df["load_pressure"]
].copy()

In [9]:
pressure_records[
    [
        "Date",
        "total_system_load",
        "net_intake"
    ]
].head(20)

,Date,total_system_load,net_intake
21,2023-02-21,8281.0,4.0
130,2023-08-01,8349.0,5.0
250,2024-01-30,8380.0,21.0
254,2024-02-05,8338.0,103.0
255,2024-02-06,8500.0,145.0
256,2024-02-07,8613.0,108.0
257,2024-02-08,8687.0,43.0
259,2024-02-12,8522.0,206.0
260,2024-02-13,8717.0,189.0
261,2024-02-14,8834.0,42.0


In [10]:
latest_load = df.iloc[-1]["total_system_load"]

In [11]:
latest_net_intake = df.iloc[-1]["net_intake"]

In [14]:
load_volatility = df["daily_load_change"].std()

In [13]:
average_load = df["total_system_load"].mean()

In [15]:
quality_issues = df["data_quality_flag"].sum()